# E9 Multistage Training

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the graph before expecting it to serve the LLM with **multiplicative** GREPs, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN
The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

## Sparse Graph Transformer
The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

## Transformer
The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM
The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

In [ ]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [ ]:
# Import modules.
import gc
import torch
import random
import numpy as np
import sympy as sp
from torch import nn
import networkx as nx

from prism.models import inference, gnn_llm, gt, r_pearl, loaders
from prism.eval import evaluate
from prism.data import data

In [ ]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 0, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [ ]:
# Standard options.
checkpoint = '../outputs/e8_new_base_models/e8_graph_mask_llm_gemma-4-12b-it_r16_4bit_v40uxzgf/'
eval_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
include_edge_list = False
use_pretrained = True
device = 'cuda:0'

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Setup the Gemma 4 model from Hugging Face.
llm = AutoModelForCausalLM.from_pretrained("google/gemma-4-12B-it", dtype="auto", device_map="auto")

# Initialize a barebones/pretrained planner for testing.
if use_pretrained:
    model = gnn_llm.GraphMaskLLM(llm, use_edges=True)
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-4-12B-it")
else:
    model, tokenizer = loaders.graph_augmented_llm_from_pretrained(
        checkpoint, load_in_4bit=True, device=device,
    )

planner = inference.GraphAugmentedInMemoryLLM(model, tokenizer, include_edge_list)

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [ ]:
from datasets import load_dataset

# Load in training dataset.
full_dataset = load_dataset("json", data_files=["../data/revised/gen/nav100_n30_gemma_data/split/formatted_all_new_2turn__train.json"], split="train")
full_dataset = data.preprocess_dataset(
    full_dataset, tokenizer,
    architecture="graph_mask_llm",
    text_edge_list=include_edge_list,
)

In [ ]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [ ]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_019.html


## Experiments

### §1 Testing a Pretrained `GraphMaskLLM`
The goal here is to test the model without any modifications to its internal architecture. By retrieving the necessary tensors for comparison, we can compute the perturbation of the attention logits and softmax distributions of the following matrix with $\mathbf{M}$ representing the block-matrix containing the graph adjacency $\mathbf{A} = \Big[\mathbb{I}\big((u, v) \in \mathcal{E}\big)\Big]_{u, v \in \mathcal{V}}$:
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot \mathbf{M}\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1}\right) \odot \mathbf{M}\right]}\right]^\top_{t \in [N]}$$

In [ ]:
"""
results = evaluate.eval_model_multiple_graphs(
    model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)
results[graph_file].path_metrics
"""

'\nresults = evaluate.eval_model_multiple_graphs(\n    model, tokenizer, eval_data,\n    include_edge_list=include_edge_list,\n    use_icl=False,\n    permutation=None,\n    on_graph_done=None\n)\nresults[graph_file].path_metrics\n'

### §2 Testing a Pretrained `GraphMaskLLM` with PE Injection into M (No Fintuning)
We would now like to instantiate a GNN and train it to replicate the graph adjacency matrix $\mathbf{A}$, defined above, to see if we achieve similar results.

In [ ]:
# Instantiate a GNN.
model_type = 'r_pearl'
if model_type == 'gt':
    gnn = gt.GraphTransformer(
        num_layers=3,
        pe_hidden_channels=256,
        pe_num_layers=2,
        d_model=1024,
        heads=8,
        num_samples=40,
        dropout=0.1,
        k_pe=2,
        k_gt=2,
        eps=1e-6,
        use_layer_norm=True
    )
else:
    gnn = r_pearl.RandomGNNPositionalEncodings(
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        num_samples=40,
        dropout=0.1,
        k=3,
        eps=1e-6,
        use_layer_norm=True
    )

#### Numeric Visualizations with SymPy
Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the eigenbasis of the graph adjacency given a scene graph PyTorch `Data` object.

In [ ]:
# Prepare a graph from the data to be used in the GNN.
from torch_geometric.utils import to_dense_adj
from prism.data import utils
import numpy as np
import sympy as sp

graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj_list = model._node_adjacency(graph, device=device).to(torch.float32)
adj = to_dense_adj(graph.edge_index).squeeze().cuda()

# Take the difference between the _node_adjacency() matrix and the to_dense_adj() matrix (for debugging).
render_matrix(adj_list.int() - adj)

Matrix([
[1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0

In [ ]:
_, eigh_vec = torch.linalg.eigh(adj_list)
render_matrix(eigh_vec, sig_figs=3)

Matrix([
[-0.0314,   0.0259, -0.0114,  -0.0025,  -0.0998,   0.0955,  -0.347, -0.0578,    0.103,        0,  0.0677,  -0.0621,    -0.21,   0.0281,        0,    0.707,    -0.111,  -0.0236,  -0.0223,  -0.133,  -0.242,   0.333,   -0.133,  -0.0735, -0.0266,    0.234, -0.0808,  0.0274, -0.0731,  -0.011],
[ 0.0565,   0.0119, -0.0577, -0.00934,    0.203,   -0.128,  0.0533, -0.0637,    0.132, -1.99e-7,   0.107,   -0.103,   -0.529,    0.446,  8.68e-7,  9.39e-7,    -0.083,   -0.101,   -0.177,   0.218,   0.257,  -0.347,   7.6e-7,   -0.217, -0.0332,    0.258,  -0.102, -0.0273, -0.0472,  -0.012],
[-0.0503,  -0.0876, 0.00465,    0.149,   0.0817,  -0.0227, -0.0668, -0.0938,   -0.168, -2.02e-7,  -0.459,   -0.279,  -0.0392, -0.00961,  4.47e-7, -1.49e-7,      0.03,   -0.567,    0.144, -0.0546,  -0.133,  -0.207,   -0.266,    0.286,  -0.233, -0.00281,  -0.109,  0.0351,  0.0231, -0.0485],
[-0.0234,   -0.113,  0.0947,   0.0456,     0.12,   0.0116,  -0.106,   0.276,    0.139, -6.62e-8,  0.0805,    0.346,   0.0

In [ ]:
# Feed the matrix to the GNN.
out = gnn(graph).to(device)
out = out - out.mean(dim=1, keepdim=True)
out = out @ out.T / torch.linalg.matrix_norm(out)
render_matrix(torch.sigmoid(out), 3)

Matrix([
[0.988, 0.988, 0.988, 0.988, 0.988, 0.988, 0.986, 0.988, 0.987, 0.987, 0.988, 0.988, 0.987, 0.988, 0.988, 0.987, 0.987, 0.987, 0.987, 0.987, 0.986, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.988, 0.988],
[0.988, 0.988, 0.988, 0.988, 0.988, 0.988, 0.986, 0.988, 0.987, 0.987, 0.988, 0.988, 0.987, 0.988, 0.988, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.988, 0.988],
[0.988, 0.988, 0.988, 0.988, 0.988, 0.988, 0.985, 0.987, 0.986, 0.986, 0.987, 0.987, 0.987, 0.987, 0.987, 0.986, 0.986, 0.986, 0.987, 0.987, 0.986, 0.986, 0.987, 0.987, 0.986, 0.986, 0.987, 0.987, 0.988, 0.987],
[0.988, 0.988, 0.988, 0.988, 0.988, 0.988, 0.985, 0.987, 0.986, 0.986, 0.987, 0.987, 0.986, 0.987, 0.987, 0.986, 0.986, 0.986, 0.986, 0.987, 0.985, 0.986, 0.987, 0.987, 0.986, 0.986, 0.986, 0.986, 0.988, 0.987],
[0.988, 0.988, 0.988, 0.988, 0.988, 0.987, 0.986, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.987, 0.988, 0.987, 0.987, 0.987, 0.987, 0.988, 0.

In [ ]:
# Compute the loss.
loss_fn = nn.CrossEntropyLoss()
loss_fn(out, adj_list).squeeze()

tensor(17.3508, device='cuda:0', grad_fn=<SqueezeBackward0>)

#### Pre-Training of GNN on Eigenbases
Next, we actually preprocess and train the GNN using the steps defined above.

In [ ]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)

# Add eigenbasis training outputs for reconstruction.
graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in train_dataset.items()]
adjs = [model._node_adjacency(graph, device=device).to(torch.float32) for graph in graphs]
eigh_vecs = [torch.linalg.eigh(adj)[1] for adj in adjs]

for i, graph in enumerate(graphs):
    graph.y = adjs[i]
    graph.x.to(device)
    graph.edge_index.to(device)
    graph.eigh_vecs = eigh_vecs[i]

In [ ]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader

batch_size = 20
epochs = 100

def train_loop(dataloader, model, loss_fn, optimizer, scheduler, batch_size=20, epochs=50):
    size = len(dataloader.dataset)
    model.to(device)
    model.train()
    for i in range(epochs):
        print(f"=============\nEpoch #{i}\n=============")
        for j, graph in enumerate(dataloader.dataset):
            # Compute prediction and loss.
            pred = model(graph)
            pred = pred - pred.mean(dim=1, keepdim=True)
            pred = pred @ pred.T / torch.linalg.matrix_norm(pred)
            loss = loss_fn(pred.to(device), graph.y)

            # Backpropagation.
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step(loss)

            # Results.
            if j % batch_size == 0:
                loss, current = loss.item(), j
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


loss_fn = nn.CrossEntropyLoss()
data_loader = DataLoader(graphs, batch_size=batch_size)
optimizer = torch.optim.AdamW(gnn.parameters(), lr=3e-7)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=20)
train_loop(data_loader, gnn, loss_fn, optimizer, None, batch_size=batch_size, epochs=epochs)

Epoch #0
Loss: 18.812574  [    0/   40]
Loss: 18.272032  [   20/   40]
Epoch #1
Loss: 18.799162  [    0/   40]
Loss: 18.302757  [   20/   40]
Epoch #2
Loss: 18.823090  [    0/   40]
Loss: 18.296028  [   20/   40]
Epoch #3
Loss: 18.809364  [    0/   40]
Loss: 18.283335  [   20/   40]
Epoch #4
Loss: 18.787939  [    0/   40]
Loss: 18.275455  [   20/   40]
Epoch #5
Loss: 18.806770  [    0/   40]
Loss: 18.292255  [   20/   40]
Epoch #6
Loss: 18.789448  [    0/   40]
Loss: 18.291832  [   20/   40]
Epoch #7
Loss: 18.807901  [    0/   40]
Loss: 18.260811  [   20/   40]
Epoch #8
Loss: 18.789211  [    0/   40]
Loss: 18.238441  [   20/   40]
Epoch #9
Loss: 18.745432  [    0/   40]
Loss: 18.254784  [   20/   40]
Epoch #10
Loss: 18.709433  [    0/   40]
Loss: 18.247322  [   20/   40]
Epoch #11
Loss: 18.712740  [    0/   40]
Loss: 18.232950  [   20/   40]
Epoch #12
Loss: 18.694777  [    0/   40]
Loss: 18.223606  [   20/   40]
Epoch #13
Loss: 18.731766  [    0/   40]
Loss: 18.180387  [   20/   40]
Ep

In [ ]:
torch.save(gnn.state_dict(), '../outputs/e9_multistage_training/eigenbasis_rpearl_normed.pt')
# gnn.load_state_dict(torch.load('../outputs/e9_multistage_training/eigenbasis_gt.pt'))

#### Evaluation of Pre-Trained GNN on Eigenbases
We now test the trained model on the evaluation dataset. First, we we will render the matrices to see their differences visually.

In [ ]:
# Load in training data.
test_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)

# Add eigenbasis test outputs for reconstruction.
test_graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for k, v in train_dataset.items()]
test_adjs = [model._node_adjacency(graph, device=device).to(torch.float32) for graph in graphs]
test_eigh_vecs = [torch.linalg.eigh(adj)[1] for adj in adjs]

for i, test_graph in enumerate(test_graphs):
    test_graph.y = adjs[i]
    test_graph.x.to(device)
    test_graph.edge_index.to(device)
    test_graph.eigh_vecs = eigh_vecs[i]

In [ ]:
# Prepare a graph from the data to be used in the GNN.
graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj_list = model._node_adjacency(graph, device=device).to(torch.float32)
_, eigh_vec = torch.linalg.eigh(adj_list)
render_matrix(eigh_vec, sig_figs=3)

Matrix([
[-0.0314,   0.0259, -0.0114,  -0.0025,  -0.0998,   0.0955,  -0.347, -0.0578,    0.103,        0,  0.0677,  -0.0621,    -0.21,   0.0281,        0,    0.707,    -0.111,  -0.0236,  -0.0223,  -0.133,  -0.242,   0.333,   -0.133,  -0.0735, -0.0266,    0.234, -0.0808,  0.0274, -0.0731,  -0.011],
[ 0.0565,   0.0119, -0.0577, -0.00934,    0.203,   -0.128,  0.0533, -0.0637,    0.132, -1.99e-7,   0.107,   -0.103,   -0.529,    0.446,  8.68e-7,  9.39e-7,    -0.083,   -0.101,   -0.177,   0.218,   0.257,  -0.347,   7.6e-7,   -0.217, -0.0332,    0.258,  -0.102, -0.0273, -0.0472,  -0.012],
[-0.0503,  -0.0876, 0.00465,    0.149,   0.0817,  -0.0227, -0.0668, -0.0938,   -0.168, -2.02e-7,  -0.459,   -0.279,  -0.0392, -0.00961,  4.47e-7, -1.49e-7,      0.03,   -0.567,    0.144, -0.0546,  -0.133,  -0.207,   -0.266,    0.286,  -0.233, -0.00281,  -0.109,  0.0351,  0.0231, -0.0485],
[-0.0234,   -0.113,  0.0947,   0.0456,     0.12,   0.0116,  -0.106,   0.276,    0.139, -6.62e-8,  0.0805,    0.346,   0.0

In [ ]:
# Feed the matrix to the GNN.
out = gnn(graph).to(device)
out = out - out.mean(dim=1, keepdim=True)
out = out @ out.T / torch.linalg.matrix_norm(out)
render_matrix(torch.sigmoid(out), 3)

Matrix([
[0.988, 0.976, 0.842, 0.749, 0.962, 0.714, 0.893, 0.951, 0.943, 0.929, 0.928, 0.934, 0.963, 0.935, 0.568, 0.639, 0.318, 0.503, 0.337, 0.934, 0.482, 0.311, 0.789, 0.732, 0.807, 0.846, 0.782, 0.756, 0.924, 0.826],
[0.976, 0.988,  0.78, 0.589, 0.971, 0.497,  0.92, 0.966, 0.962,  0.95, 0.961, 0.954, 0.974, 0.943, 0.493, 0.569, 0.192, 0.378,  0.18,  0.95, 0.382,  0.21, 0.762, 0.738, 0.769, 0.832, 0.726, 0.698, 0.887, 0.783],
[0.842,  0.78, 0.988,  0.98, 0.909, 0.974, 0.351, 0.567, 0.612, 0.435, 0.466, 0.471, 0.617, 0.407, 0.945, 0.934, 0.849,  0.91, 0.852, 0.909, 0.894, 0.843, 0.851, 0.866, 0.871, 0.907, 0.874,  0.87, 0.967, 0.923],
[0.749, 0.589,  0.98, 0.988, 0.819, 0.983, 0.256, 0.443, 0.454, 0.323, 0.321,  0.36, 0.395, 0.316, 0.948, 0.943, 0.906, 0.936, 0.909, 0.841,  0.92, 0.895, 0.837, 0.852, 0.868, 0.894, 0.872,  0.87, 0.967,  0.92],
[0.962, 0.971, 0.909, 0.819, 0.988, 0.774, 0.881, 0.943, 0.947, 0.916, 0.923, 0.913, 0.959, 0.896, 0.772, 0.821, 0.446, 0.697, 0.423, 0.978, 0.

In [ ]:
# Compute the loss.
loss_fn = nn.CrossEntropyLoss()
loss_fn(out, adj_list).squeeze()

tensor(13.9014, device='cuda:0', grad_fn=<SqueezeBackward0>)

In [ ]:
# Evaluate the GNN on its reconstruction of eigenvectors of test graph adjacencies.
def test_loop(dataloader, model, loss_fn):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            pred = model(graph)
            pred = pred - pred.mean(dim=1, keepdim=True)
            pred = pred @ pred.T / torch.linalg.matrix_norm(pred)
            test_loss += loss_fn(pred.to(device), graph.y).item()
            correct += torch.allclose(pred, graph.y)

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

del train_dataset, data, graphs, adjs, eigh_vecs
gc.collect()
data = DataLoader(test_graphs, batch_size=5)
test_loop(data, gnn, loss_fn)

Test Error: 
 Accuracy: 0.0%, Avg loss: 81.688685 



#### Performing the Injection with the Evaluated Pre-Trained GNN on Eigenbases
We now subclass the `GraphMaskLLM` class to replicate the adjacency matrix using our trained GNN instead of through the original `_node_adjacency()` method defined in the superclass. Such a system will serve as a proof of concept for the next experiment.

In [ ]:
# Define and instantiate the PEInjectionGraphMaskLLM class.
from prism.models.gnn_llm import GraphMaskLLM


class PEInjectionGraphMaskLLM(GraphMaskLLM):
    def build_structural_mask(self, seq_len, graphs, injection_maps, device, dtype=None):
        """Additive attention bias ``[B, 1, seq, seq]`` — 0 allowed, ``finfo.min`` blocked.

        ``bias[b,0,i,j] = finfo.min`` iff tokens i and j BOTH belong to graph nodes
        AND those nodes are non-adjacent (within ``k_hops``). Every other entry
        (node↔non-node, non-node↔non-node, same node, adjacent) stays 0. Because it
        is ADDED to the model's causal/sliding mask, blocking only ever removes
        already-causal pairs. Each node-token row keeps BOS (a non-node) and its own
        diagonal, so no row is fully masked (no softmax NaN).
        """
        if dtype is None:
            dtype = self.llm.get_input_embeddings().weight.dtype
        B = len(injection_maps)
        neg = torch.finfo(dtype).min
        bias = torch.zeros(B, 1, seq_len, seq_len, device=device, dtype=dtype)
        for b in range(B):
            g = graphs[b]
            # token position -> node id (-1 for non-node tokens). Spans are disjoint
            # (build_injection_map dedups longest-first), so each token maps to one node.
            tok2node = torch.full((seq_len,), -1, dtype=torch.long, device=device)
            for node_idx, spans in injection_maps[b].items():
                for start, end in spans:
                    end = min(end, seq_len)
                    if start < end:
                        tok2node[start:end] = node_idx
            node_pos = (tok2node >= 0).nonzero(as_tuple=True)[0]
            if node_pos.numel() == 0:
                continue
            eigh_vecs = gnn(g)
            adj = torch.sigmoid(eigh_vecs @ eigh_vecs.T / torch.linalg.norm(eigh_vecs)).round().int()
            nid = tok2node[node_pos]
            allowed = adj[nid][:, nid]
            blocked = ~allowed
            if blocked.any():
                bi, bj = blocked.nonzero(as_tuple=True)
                bias[b, 0, node_pos[bi], node_pos[bj]] = neg
        return bias

In [ ]:
new_model = PEInjectionGraphMaskLLM(llm)
new_model.load_state_dict(model.state_dict())

"""
results = evaluate.eval_model_multiple_graphs(
    new_model, tokenizer, eval_data,
    include_edge_list=include_edge_list,
    use_icl=False,
    permutation=None,
    on_graph_done=None
)
results[graph_file].path_metrics
"""

'\nresults = evaluate.eval_model_multiple_graphs(\n    new_model, tokenizer, eval_data,\n    include_edge_list=include_edge_list,\n    use_icl=False,\n    permutation=None,\n    on_graph_done=None\n)\nresults[graph_file].path_metrics\n'